# 02. EDA: реальная выгрузка инвойсов как прокси биллинга data room

**Зачем.** Синтетика (P2) рискует быть чище реального data room (риск R2). Здесь мы берём реальную выгрузку
построчных инвойсов и меряем, какая «грязь» в ней есть и насколько сильно она меняет KPI. Результат —
таблица «измеренная частота → параметр шума генератора» (раздел 10) для `configs/synth.yaml`.

**Датасет.** UCI Online Retail II (Chen, 2019), CC BY 4.0,
<https://archive.ics.uci.edu/dataset/502/online+retail+ii>. Онлайн-ритейлер подарков из Великобритании,
B2C/B2B-оптовики, 01.12.2009–09.12.2011, два листа Excel. Скачивание и кэш в parquet:
`uv run python scripts/fetch_online_retail.py`.

**Как читать.** Мы смотрим на датасет глазами FR1 («загрузка в DuckDB с проверкой схемы и качества»):
строка = строка инвойса, `Invoice` с префиксом `C` = отмена (аналог кредит-ноты), `Customer ID` = ключ
к справочнику клиентов. Выводы — в `reports/eda/02_billing_proxy.md`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "external" / "online_retail_ii"
FIG = ROOT / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

# Палитра (категориальная, фиксированный порядок) и чернила для текста
C1, C2, C3, C4 = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": INK2,
        "axes.labelcolor": INK,
        "axes.titlesize": 12,
        "axes.titleweight": "bold",
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "axes.axisbelow": True,
        "xtick.color": INK2,
        "ytick.color": INK2,
        "font.size": 10,
    }
)


def save(fig, name):
    path = FIG / f"02_{name}.png"
    fig.savefig(path, dpi=110, bbox_inches="tight")
    print(f"{path.relative_to(ROOT)}: {path.stat().st_size / 1024:.0f} KB")


def pct(x):
    return f"{100 * x:.2f}%"

## 1. Загрузка, объём, схема

Два листа — две выгрузки за «финансовые годы» (декабрь–ноябрь). Первое, что видно на реальных данных:
выгрузки **перекрываются по датам**. Проверим, насколько.

In [ ]:
raw_a = pd.read_parquet(DATA / "online_retail_ii_2009_2010.parquet").assign(sheet="2009-2010")
raw_b = pd.read_parquet(DATA / "online_retail_ii_2010_2011.parquet").assign(sheet="2010-2011")
for r in (raw_a, raw_b):
    print(r["sheet"].iat[0], f"{len(r):,} строк", r.InvoiceDate.min(), "—", r.InvoiceDate.max())
print(raw_a.dtypes)

cols = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country",
]
overlap_start = raw_b.InvoiceDate.min().normalize()
ov_a = raw_a[raw_a.InvoiceDate >= overlap_start]
ov_b = raw_b[raw_b.InvoiceDate <= raw_a.InvoiceDate.max()]
same = (
    ov_a[cols]
    .sort_values(cols)
    .reset_index(drop=True)
    .equals(ov_b[cols].sort_values(cols).reset_index(drop=True))
)
print(
    f"перекрытие {overlap_start.date()}–{raw_a.InvoiceDate.max().date()}: "
    f"{len(ov_a):,} строк / {ov_a.Invoice.nunique():,} инвойсов в каждом листе, совпадают построчно: {same}"
)
ov_amount = ov_a.Quantity * ov_a.Price
print(
    f"сумма в перекрытии: нетто £{ov_amount.sum():,.0f}; "
    f"валовая сумма положительных строк £{ov_amount[ov_amount > 0].sum():,.0f}"
)

In [ ]:
naive = pd.concat([raw_a, raw_b], ignore_index=True)  # «как пришло»: два файла склеены
df = pd.concat([raw_a[raw_a.InvoiceDate < overlap_start], raw_b], ignore_index=True)
df["amount"] = df.Quantity * df.Price
df["month"] = df.InvoiceDate.dt.to_period("M")
df["cust"] = df["Customer ID"]

overview = pd.Series(
    {
        "строк (склейка двух листов)": len(naive),
        "строк (без перекрытия)": len(df),
        "инвойсов": df.Invoice.nunique(),
        "клиентов (Customer ID)": df.cust.nunique(),
        "товарных кодов": df.StockCode.nunique(),
        "стран": df.Country.nunique(),
        "период, мес.": df.month.nunique(),
        "первая дата": df.InvoiceDate.min(),
        "последняя дата": df.InvoiceDate.max(),
    }
)
overview

## 2. Типы строк: товар, отмена, служебные строки

В биллинге SaaS у строки есть `payment_type` и `status`. Здесь их нет, и тип приходится выводить из
префикса инвойса (`C` — отмена, `A` — корректировка долга) и `StockCode` (служебные коды вроде `POST`, `M`, `D`).
Это и есть типичная работа на шаге «приём данных»: восстановить тип строки по косвенным признакам.

In [ ]:
SERVICE_CODES = {
    "POST": "доставка",
    "DOT": "доставка",
    "C2": "доставка",
    "C3": "доставка",
    "M": "ручная строка (Manual)",
    "m": "ручная строка (Manual)",
    "D": "скидка",
    "S": "образцы",
    "BANK CHARGES": "комиссии",
    "AMAZONFEE": "комиссии",
    "CRUK": "комиссии",
    "B": "списание долга",
    "ADJUST": "корректировка",
    "ADJUST2": "корректировка",
    "TEST001": "тестовая строка",
    "TEST002": "тестовая строка",
}


def service_kind(code):
    if code in SERVICE_CODES:
        return SERVICE_CODES[code]
    if code.startswith("gift_0001"):
        return "подарочный сертификат"
    return None


df["service"] = df.StockCode.map(service_kind)
df["is_service"] = df.service.notna()
df["prefix"] = np.where(df.Invoice.str[0].str.isdigit(), "обычный", df.Invoice.str[0])
df["is_cancel"] = df.prefix == "C"


def line_type(r):
    if r.prefix == "A":
        return "корректировка долга (A)"
    if r.is_cancel:
        return "отмена (C)"
    if r.is_service:
        return "служебная строка"
    if r.Price == 0:
        return "нулевая цена"
    if r.Quantity < 0:
        return "складская корректировка (qty<0)"
    return "товар"


df["line_type"] = [
    line_type(r)
    for r in df[["prefix", "is_cancel", "is_service", "Price", "Quantity"]].itertuples()
]
lt = df.groupby("line_type").agg(
    rows=("Invoice", "size"), invoices=("Invoice", "nunique"), amount=("amount", "sum")
)
lt["rows_share"] = lt.rows / len(df)
lt["amount_share_of_gross"] = lt.amount / df.loc[df.amount > 0, "amount"].sum()
lt.sort_values("rows", ascending=False)

In [ ]:
order = lt.sort_values("rows").index
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
axes[0].barh(order, 100 * lt.loc[order, "rows_share"], color=C1, height=0.6)
axes[0].set_xscale("log")
axes[0].set_xlabel("доля строк, % (лог. шкала)")
axes[0].set_title("Типы строк: доля строк")
for y, (v, n) in enumerate(zip(lt.loc[order, "rows_share"], lt.loc[order, "rows"], strict=True)):
    lbl = f"{100 * v:.2f}%" if v >= 1e-4 else f"{n} строк"
    axes[0].text(100 * v * 1.15, y, lbl, va="center", fontsize=8, color=INK2)
vals = 100 * lt.loc[order, "amount_share_of_gross"]
axes[1].barh(order, vals, color=[C2 if v < 0 else C1 for v in vals], height=0.6)
axes[1].axvline(0, color=INK2, lw=0.8)
for y, v in enumerate(vals):
    axes[1].text(
        v + (1.5 if v >= 0 else -1.5),
        y,
        f"{v:+.1f}%",
        va="center",
        ha="left" if v >= 0 else "right",
        fontsize=8,
        color=INK2,
    )
axes[1].set_xlim(-20, 115)
axes[1].set_xlabel("сумма Quantity × Price, % валовой выручки")
axes[1].set_title("Типы строк: доля суммы")
fig.suptitle(
    "Online Retail II, 12.2009–12.2011, без перекрытия выгрузок", fontsize=10, color=INK2, y=1.02
)
fig.tight_layout()
save(fig, "line_types")

## 3. Служебные (нетоварные) строки

Аналог в SaaS: разовые платежи, внедрение, возмещаемые расходы, ручные корректировки, скидки. Именно они
попадают в ARR при D01 (`incl_one_time`). Меряем частоту и вклад в выручку.

In [ ]:
svc = df[df.is_service & (df.prefix != "A")]
svc_tab = svc.groupby("service").agg(
    rows=("Invoice", "size"),
    invoices=("Invoice", "nunique"),
    plus=("amount", lambda s: s[s > 0].sum()),
    minus=("amount", lambda s: s[s < 0].sum()),
)
gross_product = df.loc[(df.line_type == "товар"), "amount"].sum()
svc_tab["plus_share_of_product_gross"] = svc_tab.plus / gross_product
svc_tab = svc_tab.sort_values("plus", ascending=False)
inv_with_svc = (
    df.loc[~df.is_cancel & df.is_service, "Invoice"].nunique()
    / df.loc[~df.is_cancel, "Invoice"].nunique()
)
print(
    f"служебных строк: {len(svc):,} ({pct(len(svc) / len(df))} строк); без отмен: "
    f"{int((~svc.is_cancel).sum()):,} ({pct((~svc.is_cancel).sum() / len(df))}), "
    f"в отменах: {int(svc.is_cancel.sum()):,}"
)
print(f"доля обычных инвойсов, где есть служебная строка: {pct(inv_with_svc)}")
print(f"плюс от служебных строк к товарной выручке: {pct(svc_tab.plus.sum() / gross_product)}")
print(
    f"плюс от служебных строк к валовой сумме всех строк: {pct(svc_tab.plus.sum() / df.loc[df.amount > 0, 'amount'].sum())}"
)
svc_tab

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
t = svc_tab.sort_values("plus")
ax.barh(t.index, t.plus / 1e3, color=C1, height=0.6, label="положительные суммы")
ax.barh(t.index, t.minus / 1e3, color=C2, height=0.6, label="отрицательные суммы")
ax.axvline(0, color=INK2, lw=0.8)
ax.set_xlabel("тыс. £ за 12.2009–12.2011")
ax.set_title("Служебные строки: что попадёт в «выручку», если не фильтровать")
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
save(fig, "non_product_lines")

## 4. Отмены (`C`) как кредит-ноты

В выгрузке нет ссылки отмены на исходный инвойс — как и во многих реальных биллингах, где кредит-нота
выписывается отдельным документом. Сопоставляем эвристикой: ближайшая предыдущая покупка того же клиента
и того же товара (`merge_asof`). Важны две вещи: доля отмен и **лаг** — кредит-нота часто попадает в другой
месяц, чем исходная продажа, и тогда помесячная выручка (и MRR) зависит от того, к какому месяцу её отнести.

In [ ]:
cancel = df[df.is_cancel]
pos_inv = df[(df.prefix == "обычный")]
n_inv_regular = pos_inv.Invoice.nunique()
gross_pos = df.loc[df.amount > 0, "amount"].sum()
print(
    f"инвойсов-отмен: {cancel.Invoice.nunique():,} = {pct(cancel.Invoice.nunique() / df.Invoice.nunique())} всех инвойсов"
)
print(f"строк-отмен: {len(cancel):,} = {pct(len(cancel) / len(df))} строк")
print(
    f"сумма отмен: £{cancel.amount.sum():,.0f} = {pct(-cancel.amount.sum() / gross_pos)} валовой выручки"
)
print(f"отмен без Customer ID: {pct(cancel.cust.isna().mean())} строк")
print(
    f"клиентов с ≥1 отменой: {cancel.cust.nunique():,} из {df.cust.nunique():,} ({pct(cancel.cust.nunique() / df.cust.nunique())})"
)
print("отмены по типу строки:")
print(cancel.service.fillna("товар").value_counts().head(8))

In [ ]:
c = cancel.dropna(subset=["cust"]).loc[lambda x: ~x.is_service].sort_values("InvoiceDate")
o = (
    df[(df.prefix == "обычный") & df.cust.notna() & (df.Quantity > 0)]
    .sort_values("InvoiceDate")[
        ["InvoiceDate", "cust", "StockCode", "Quantity", "Invoice", "month"]
    ]
    .rename(
        columns={
            "InvoiceDate": "orig_date",
            "Quantity": "orig_qty",
            "Invoice": "orig_inv",
            "month": "orig_month",
        }
    )
)
m = pd.merge_asof(
    c[["InvoiceDate", "cust", "StockCode", "Quantity", "amount", "month"]],
    o.assign(InvoiceDate=o.orig_date),
    on="InvoiceDate",
    by=["cust", "StockCode"],
    direction="backward",
)
m["matched"] = m.orig_inv.notna()
m["lag_days"] = (m.InvoiceDate - m.orig_date).dt.total_seconds() / 86400
m["exact_qty"] = m.orig_qty == -m.Quantity
m["cross_month"] = m.matched & (m.month != m.orig_month)
mm = m[m.matched]
print(
    f"сопоставлено с исходной продажей: {pct(m.matched.mean())} строк-отмен (товарных, с Customer ID)"
)
print(f"  из них то же количество: {pct(mm.exact_qty.mean())}")
print(
    f"  лаг, дней: медиана {mm.lag_days.median():.1f}, p75 {mm.lag_days.quantile(0.75):.1f}, p90 {mm.lag_days.quantile(0.9):.1f}"
)
print(
    f"  отмена в другом календарном месяце: {pct(mm.cross_month.mean())} строк, {pct(mm.loc[mm.cross_month, 'amount'].sum() / mm.amount.sum())} суммы"
)
print(f"  отмена позже 30 дней: {pct((mm.lag_days > 30).mean())}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
bins = np.r_[0, 1, 2, 3, 7, 14, 30, 60, 90, 180, 365, 740]
h, _ = np.histogram(mm.lag_days, bins=bins)
labels = ["<1", "1–2", "2–3", "3–7", "7–14", "14–30", "30–60", "60–90", "90–180", "180–365", ">365"]
share = h / h.sum()
ax.bar(labels, 100 * share, color=[C1 if b < 30 else C2 for b in bins[:-1]], width=0.7)
for x, v in enumerate(share):
    ax.text(x, 100 * v + 0.5, f"{100 * v:.0f}%", ha="center", fontsize=8, color=INK2)
ax.set_xlabel("дней от исходной продажи до отмены")
ax.set_ylabel("доля строк-отмен, %")
ax.set_title(f"Лаг отмен: {100 * mm.cross_month.mean():.0f}% строк-отмен попадают в другой месяц")
fig.tight_layout()
save(fig, "cancellation_lag")

### 4.1. Отрицательные количества без префикса `C` и нулевые/отрицательные цены

In [ ]:
neg_nonc = df[(df.Quantity < 0) & ~df.is_cancel]
zero_p = df[df.Price == 0]
neg_p = df[df.Price < 0]
print(
    f"qty<0 без префикса C: {len(neg_nonc):,} строк ({pct(len(neg_nonc) / len(df))}), "
    f"с Customer ID: {neg_nonc.cust.notna().sum()}, цена 0: {pct((neg_nonc.Price == 0).mean())}"
)
print(
    f"нулевая цена: {len(zero_p):,} строк ({pct(len(zero_p) / len(df))}), без Customer ID: {pct(zero_p.cust.isna().mean())}"
)
print(f"отрицательная цена: {len(neg_p)} строк, сумма £{neg_p.amount.sum():,.0f}")
print(
    neg_p[["Invoice", "StockCode", "Description", "Quantity", "Price", "InvoiceDate"]].to_string()
)
print("\nтоп описаний строк с нулевой ценой (складские пометки, не продажи):")
print(zero_p.Description.str.lower().value_counts().head(10).to_string())

## 5. Пропущенный `Customer ID`

В SaaS это инвойс, который нельзя связать со справочником клиентов: его выручка есть, а клиента нет.
Для метрик это значит: выручка и MRR не совпадут с суммой по клиентам, а клиентские метрики (churn, NRR,
концентрация) считаются на неполной базе.

In [ ]:
na = df.cust.isna()
mixed = df.groupby("Invoice").cust.agg(lambda s: s.isna().any() and s.notna().any()).sum()
prod = df.line_type == "товар"
miss = pd.Series(
    {
        "строк": na.mean(),
        "инвойсов": df.loc[na, "Invoice"].nunique() / df.Invoice.nunique(),
        "валовой выручки (все строки > 0)": df.loc[na & (df.amount > 0), "amount"].sum()
        / gross_pos,
        "товарной выручки": df.loc[na & prod, "amount"].sum() / df.loc[prod, "amount"].sum(),
        "суммы отмен": cancel.loc[cancel.cust.isna(), "amount"].sum() / cancel.amount.sum(),
    },
    name="доля без Customer ID",
)
print(f"инвойсов, где ID есть только у части строк: {mixed} (пропуск — на уровне всего инвойса)")
print(miss.map(pct).to_string())
print("\nстраны строк без ID:")
print(df.loc[na, "Country"].value_counts(normalize=True).head(5).map(pct).to_string())
big = df[na].groupby("Invoice").agg(lines=("StockCode", "size"), amount=("amount", "sum"))
print(
    f"\nинвойсы без ID: медиана строк {big.lines.median():.0f} vs {df[~na].groupby('Invoice').size().median():.0f} у инвойсов с ID"
)

In [ ]:
mon = df.groupby("month").apply(
    lambda g: pd.Series(
        {
            "rows": g.cust.isna().mean(),
            "rev": g.loc[g.cust.isna() & (g.amount > 0), "amount"].sum()
            / g.loc[g.amount > 0, "amount"].sum(),
        }
    ),
    include_groups=False,
)
fig, ax = plt.subplots(figsize=(9, 3.4))
x = mon.index.to_timestamp()
ax.plot(x, 100 * mon.rows, color=C1, lw=2, marker="o", ms=4, label="доля строк")
ax.plot(x, 100 * mon.rev, color=C2, lw=2, marker="o", ms=4, label="доля выручки")
ax.set_ylabel("% без Customer ID")
ax.set_ylim(0, None)
ax.set_title("Пропуск Customer ID по месяцам: непостоянен во времени")
ax.legend(frameon=False)
fig.tight_layout()
save(fig, "missing_customer_monthly")
mon.map(pct).T

## 6. Дубли: точные и почти-дубли

In [ ]:
dup_mask = df.duplicated(subset=cols, keep="first")
dup_inv = df.loc[dup_mask, "Invoice"].nunique()
print(
    f"точные дубли строк (все 8 полей): {dup_mask.sum():,} = {pct(dup_mask.mean())} строк, "
    f"в {dup_inv:,} инвойсах ({pct(dup_inv / df.Invoice.nunique())}), сумма £{df.loc[dup_mask, 'amount'].sum():,.0f} "
    f"({pct(df.loc[dup_mask, 'amount'].sum() / gross_pos)} валовой)"
)
print(
    f"перекрытие выгрузок (дубли между файлами): {len(ov_a):,} строк = {pct(len(ov_a) / len(df))} строк, "
    f"{ov_a.Invoice.nunique():,} инвойсов = {pct(ov_a.Invoice.nunique() / df.Invoice.nunique())}"
)

# Почти-дубли на уровне инвойса: тот же клиент, тот же набор строк (код, кол-во, цена), другой номер, ≤ 24 ч
sig = (
    df[df.cust.notna() & ~df.is_cancel]
    .sort_values(["Invoice", "StockCode", "Quantity", "Price"])
    .assign(key=lambda x: x.StockCode + "|" + x.Quantity.astype(str) + "|" + x.Price.astype(str))
    .groupby("Invoice")
    .agg(
        cust=("cust", "first"),
        date=("InvoiceDate", "min"),
        sig=("key", "|".join),
        total=("amount", "sum"),
        lines=("key", "size"),
    )
    .sort_values("date")
)
sig["prev_gap_h"] = sig.groupby(["cust", "sig"]).date.diff().dt.total_seconds() / 3600
nd = sig[sig.prev_gap_h <= 24]
print(
    f"почти-дубли инвойсов (тот же клиент и состав, другой номер, ≤24 ч): {len(nd):,} = {pct(len(nd) / len(sig))} инвойсов, "
    f"сумма £{nd.total.sum():,.0f} ({pct(nd.total.sum() / gross_pos)} валовой); из них однострочных {pct((nd.lines == 1).mean())}"
)
print(
    f"  ≤ 1 ч: {(sig.prev_gap_h <= 1).sum():,}; многострочных (≥3 строк) ≤ 24 ч: {(nd.lines >= 3).sum():,}"
)

# Вариативность названий: один StockCode — несколько описаний (аналог «Acme Inc.» vs «ACME LLC»)
desc = df.dropna(subset=["Description"]).groupby("StockCode").Description.nunique()
print(
    f"товарных кодов с >1 вариантом описания: {(desc > 1).sum():,} из {len(desc):,} ({pct((desc > 1).mean())})"
)
cc = df.dropna(subset=["cust"]).groupby("cust").Country.nunique()
print(f"клиентов с >1 страной: {(cc > 1).sum()} из {len(cc):,} ({pct((cc > 1).mean())})")

## 7. Даты: перекрытия, «рваные» месяцы, нерабочие дни, порядок номеров

In [ ]:
inv = df.groupby("Invoice").agg(
    date=("InvoiceDate", "min"), dmax=("InvoiceDate", "max"), prefix=("prefix", "first")
)
multi_ts = (inv.dmax > inv.date).sum()
reg = inv[inv.prefix == "обычный"].copy()
reg["num"] = reg.index.astype(int)
reg = reg.sort_values("num")
backdated = (reg.date < reg.date.cummax() - pd.Timedelta(days=1)).sum()
dow = df.InvoiceDate.dt.day_name().value_counts()
days = pd.Series(1, index=pd.DatetimeIndex(df.InvoiceDate.dt.normalize().unique())).sort_index()
gaps = days.index.to_series().diff().dt.days
last_month_days = df.loc[df.month == df.month.max(), "InvoiceDate"].dt.day.max()
print(f"инвойсов с несколькими временными метками: {multi_ts} ({pct(multi_ts / len(inv))})")
print(
    f"номера инвойсов не по порядку дат (выписаны «задним числом» > 1 дня): {backdated} из {len(reg):,}"
)
print(f"строк по дням недели:\n{dow.to_string()}")
sat = df[df.InvoiceDate.dt.day_name() == "Saturday"]
print(
    f"субботние строки: даты {sorted(sat.InvoiceDate.dt.date.unique())}, инвойсов {sat.Invoice.nunique()}"
)
print(f"самые длинные перерывы в торговле (дней): {gaps.nlargest(3).to_dict()}")
print(f"последний месяц ({df.month.max()}) неполный: данные по {last_month_days}-е число")

## 8. Страны и валюта

In [ ]:
by_c = df[prod].groupby("Country").amount.sum().sort_values(ascending=False)
print(
    f"стран: {by_c.size}; доля UK в товарной выручке: {pct(by_c.iloc[0] / by_c.sum())}; "
    f"топ-5 стран: {pct(by_c.iloc[:5].sum() / by_c.sum())}"
)
print(by_c.head(8).map(lambda v: f"£{v:,.0f}").to_string())
print("\nвсе цены в GBP: в выгрузке нет поля валюты, страна ≠ валюта")
print(
    f"строк со страной 'Unspecified'/'European Community': {df.Country.isin(['Unspecified', 'European Community']).sum()}"
)

## 9. Концентрация клиентов и помесячная динамика

Для D02 (пиковый месяц × 12) и D04 (удобная база сравнения) важна сезонность: насколько пик и провал
отличаются от среднего. Считаем на «чистой» выручке: товарные строки, нетто с отменами, без перекрытия
выгрузок и точных дублей.

In [ ]:
clean = df[~dup_mask & ((df.line_type == "товар") | (df.is_cancel & ~df.is_service))].copy()
monthly = clean.groupby("month").amount.sum()
t12 = (monthly.index >= pd.Period("2010-12", "M")) & (monthly.index <= pd.Period("2011-11", "M"))
m12 = monthly[t12]
annual = m12.sum()
peak_m, trough_m = m12.idxmax(), m12.idxmin()
print(f"чистая выручка 12.2010–11.2011: £{annual:,.0f}; средний месяц £{m12.mean():,.0f}")
print(
    f"пик {peak_m}: £{m12.max():,.0f} = {m12.max() / m12.mean():.2f}× среднего; провал {trough_m}: {m12.min() / m12.mean():.2f}×"
)
print(
    f"D02: пиковый месяц × 12 = £{12 * m12.max():,.0f} = {12 * m12.max() / annual:.2f}× фактической годовой"
)
print(f"D02: последний месяц (ноябрь) × 12 = {12 * m12.iloc[-1] / annual:.2f}× фактической годовой")
yoy = monthly[pd.Period("2011-11", "M")] / monthly[pd.Period("2010-11", "M")]
from_trough = monthly[pd.Period("2011-11", "M")] / m12.min()
print(
    f"D04: ноябрь 2011 к ноябрю 2010 (честно, YoY): ×{yoy:.2f}; к провальному {trough_m}: ×{from_trough:.2f}"
)
partial = monthly.iloc[-1] / monthly.iloc[-2] - 1
print(f"неполный декабрь 2011 к ноябрю: {100 * partial:+.0f}% (артефакт периода, не падение)")
cv = m12.std() / m12.mean()
print(f"коэффициент вариации месячной выручки: {cv:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
x = monthly.index.to_timestamp()
y = monthly.values / 1e3
ax.plot(x[:-1], y[:-1], color=C1, lw=2, marker="o", ms=4)
ax.plot(x[-2:], y[-2:], color=C1, lw=2, ls=":", marker="o", ms=4)
ax.axhline(m12.mean() / 1e3, color=INK2, lw=1, ls="--")
ax.text(x[0], m12.mean() / 1e3 * 1.03, "средний месяц 12.2010–11.2011", fontsize=8, color=INK2)
for p, lbl, dx, dy, ha in [
    (peak_m, "пик", 0, 10, "center"),
    (trough_m, "провал", 0, -32, "center"),
    (monthly.index[-1], "неполный месяц\n(1–9 дек.)", -8, -12, "right"),
]:
    v = monthly[p] / 1e3
    ax.annotate(
        f"{lbl}\n£{v:,.0f}k",
        (p.to_timestamp(), v),
        textcoords="offset points",
        xytext=(dx, dy),
        ha=ha,
        fontsize=8,
        color=INK,
    )
ax.set_ylabel("тыс. £ в месяц")
ax.set_ylim(0, None)
ax.set_title("Чистая выручка по месяцам: сильная сезонность (пик в ноябре)")
fig.tight_layout()
save(fig, "monthly_revenue")

In [ ]:
w = clean[(clean.month >= pd.Period("2010-12", "M")) & (clean.month <= pd.Period("2011-11", "M"))]
per_cust = w.dropna(subset=["cust"]).groupby("cust").amount.sum().sort_values(ascending=False)
per_cust = per_cust[per_cust > 0]
top10 = per_cust.iloc[:10].sum() / per_cust.sum()
top1 = per_cust.iloc[0] / per_cust.sum()
anon = w.loc[w.cust.isna(), "amount"].sum()
top10_all = per_cust.iloc[:10].sum() / (per_cust.sum() + anon)
print(f"клиентов с положительной выручкой: {len(per_cust):,}")
print(
    f"top-10 доля (знаменатель — выручка идентифицированных клиентов): {pct(top10)}; top-1: {pct(top1)}"
)
print(f"top-10 доля (знаменатель — вся выручка, включая без ID): {pct(top10_all)}")
print(
    f"«клиент без ID» как одна строка справочника был бы №{(per_cust > anon).sum() + 1} с £{anon:,.0f}"
)
top20pct = per_cust.iloc[: int(0.2 * len(per_cust))].sum() / per_cust.sum()
print(f"top-20% клиентов дают {pct(top20pct)} выручки")

fig, ax = plt.subplots(figsize=(6.5, 4))
cum = per_cust.cumsum() / per_cust.sum()
xs = np.arange(1, len(cum) + 1) / len(cum)
ax.plot(100 * xs, 100 * cum.values, color=C1, lw=2)
ax.plot([0, 100], [0, 100], color=INK2, lw=1, ls="--")
ax.scatter([100 * 10 / len(cum)], [100 * top10], color=C2, s=40, zorder=3)
ax.annotate(
    f"top-10 = {100 * top10:.1f}%",
    (100 * 10 / len(cum), 100 * top10),
    xytext=(8, -4),
    textcoords="offset points",
    fontsize=9,
)
ax.set_xlabel("доля клиентов (по убыванию выручки), %")
ax.set_ylabel("накопленная доля выручки, %")
ax.set_title("Концентрация выручки, 12.2010–11.2011")
fig.tight_layout()
save(fig, "customer_concentration")

## 10. Наивный vs очищенный KPI: сколько стоят определения и очистка

Окно — 12 полных месяцев 12.2010–11.2011 (декабрь 2010 есть в обоих листах). Шаги очистки применяются
последовательно; каждый шаг — отдельное решение «что считать выручкой».

In [ ]:
lo, hi = pd.Timestamp("2010-12-01"), pd.Timestamp("2011-12-01")
n_all = naive.assign(amount=naive.Quantity * naive.Price)
n_all = n_all[(n_all.InvoiceDate >= lo) & (n_all.InvoiceDate < hi)]
d12 = df[(df.InvoiceDate >= lo) & (df.InvoiceDate < hi)]
dm12 = dup_mask[d12.index]

steps = []
steps.append(
    (
        "наивно: склейка двух листов, сумма всех строк > 0 (gross)",
        n_all.loc[n_all.amount > 0, "amount"].sum(),
    )
)
steps.append(("− перекрытие выгрузок (1–9.12.2010)", d12.loc[d12.amount > 0, "amount"].sum()))
steps.append(("− отмены (нетто, вычитаем кредит-ноты)", d12.amount.sum()))
steps.append(("− точные дубли строк", d12.loc[~dm12, "amount"].sum()))
keep = ~dm12 & ((d12.line_type == "товар") | (d12.is_cancel & ~d12.is_service))
steps.append(
    (
        "− служебные строки и корректировки (доставка, Manual, комиссии, bad debt, нулевые)",
        d12.loc[keep, "amount"].sum(),
    )
)
steps.append(("− строки без Customer ID", d12.loc[keep & d12.cust.notna(), "amount"].sum()))
wf = pd.DataFrame(steps, columns=["шаг", "выручка"])
wf["к наивной"] = wf["выручка"] / wf["выручка"].iat[0] - 1
wf["дельта шага"] = wf["выручка"].diff()
print(
    wf.to_string(
        formatters={
            "выручка": "£{:,.0f}".format,
            "к наивной": "{:+.1%}".format,
            "дельта шага": "£{:,.0f}".format,
        }
    )
)
print(f"наивная / очищенная (с ID) = {wf['выручка'].iat[0] / wf['выручка'].iat[-1]:.3f}")
print(
    f"наивная / очищенная (вся товарная нетто) = {wf['выручка'].iat[0] / wf['выручка'].iat[-2]:.3f}"
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
vals = wf["выручка"].to_numpy() / 1e6
labels = [
    "наивно\n(gross, 2 листа)",
    "− перекрытие",
    "− отмены",
    "− дубли",
    "− служебные\nи корректировки",
    "− без ID",
]
prev = vals[0]
ax.bar(0, vals[0], color=C1, width=0.6)
ax.text(0, vals[0] + 0.05, f"£{vals[0]:.2f}M", ha="center", fontsize=9)
for i in range(1, len(vals)):
    ax.bar(i, vals[i] - prev, bottom=prev, color=C2 if vals[i] < prev else C3, width=0.6)
    ax.text(
        i,
        max(prev, vals[i]) + 0.05,
        f"{(vals[i] - prev) * 1e3:+,.0f}k",
        ha="center",
        fontsize=8,
        color=INK2,
    )
    prev = vals[i]
ax.bar(len(vals), vals[-1], color=C3, width=0.6)
ax.text(len(vals), vals[-1] + 0.05, f"£{vals[-1]:.2f}M", ha="center", fontsize=9)
ax.set_xticks(range(len(vals) + 1), [*labels, "очищено\n(товары, нетто, с ID)"], fontsize=8)
ax.set_ylabel("млн £ за 12.2010–11.2011")
ax.set_ylim(0, vals[0] * 1.12)
ax.set_title(
    f"Одна выгрузка — разные «выручки»: наивная больше очищенной на {100 * (vals[0] / vals[-1] - 1):.0f}%"
)
fig.tight_layout()
save(fig, "kpi_waterfall")

In [ ]:
# Число клиентов при разных определениях (то же окно)
nov = d12[d12.month == pd.Period("2011-11", "M")]
net_by_c = d12[keep].groupby("cust").amount.sum()
cust_defs = pd.Series(
    {
        "любая строка в выгрузке (вкл. только отмены)": d12.cust.nunique(),
        "≥1 покупка (обычный инвойс)": d12.loc[d12.prefix == "обычный", "cust"].nunique(),
        "нетто-выручка > 0 за 12 мес.": (net_by_c > 0).sum(),
        "≥2 покупок за 12 мес. (повторные)": d12[d12.prefix == "обычный"]
        .groupby("cust")
        .Invoice.nunique()
        .ge(2)
        .sum(),
        "активны в последнем месяце (11.2011)": nov.loc[nov.prefix == "обычный", "cust"].nunique(),
    },
    name="клиентов",
)
only_cancel = d12.groupby("cust").is_cancel.all().sum()
print(f"клиентов, у которых в окне только отмены: {only_cancel}")
print(
    (cust_defs / cust_defs.iloc[0])
    .map(pct)
    .rename("к наивному")
    .to_frame()
    .join(cust_defs)
    .to_string()
)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.barh(cust_defs.index[::-1], cust_defs.values[::-1], color=C1, height=0.6)
for y, v in enumerate(cust_defs.values[::-1]):
    ax.text(v + 30, y, f"{v:,}", va="center", fontsize=9)
ax.set_xlabel("число клиентов")
ax.set_xlim(0, cust_defs.max() * 1.15)
ax.set_title("«Сколько у нас клиентов?» — пять ответов по одной выгрузке")
fig.tight_layout()
save(fig, "customer_count_definitions")

## 11. Сводка частот для калибровки шума генератора

Все частоты — на данных без перекрытия выгрузок (кроме самой строки про перекрытие).
Пересчёт в параметры B2B SaaS — в отчёте `reports/eda/02_billing_proxy.md`, раздел «Калибровка шума».

In [ ]:
rates = pd.DataFrame(
    [
        ("точные дубли строк", "строк", dup_mask.mean()),
        ("точные дубли строк", "инвойсов", dup_inv / df.Invoice.nunique()),
        ("почти-дубли инвойсов (≤24 ч, тот же состав)", "инвойсов", len(nd) / len(sig)),
        ("перекрытие двух выгрузок", "строк", len(ov_a) / len(df)),
        ("пропуск Customer ID", "строк", na.mean()),
        ("пропуск Customer ID", "инвойсов", df.loc[na, "Invoice"].nunique() / df.Invoice.nunique()),
        ("пропуск Customer ID", "валовой выручки", miss.iloc[2]),
        ("отмены (C)", "инвойсов", cancel.Invoice.nunique() / df.Invoice.nunique()),
        ("отмены (C)", "валовой выручки", -cancel.amount.sum() / gross_pos),
        ("отмена в другом месяце", "строк-отмен", mm.cross_month.mean()),
        ("отмена позже 30 дней", "строк-отмен", (mm.lag_days > 30).mean()),
        ("служебные строки", "строк", len(svc) / len(df)),
        ("служебные строки", "обычных инвойсов", inv_with_svc),
        ("служебные строки (+)", "товарной выручки", svc_tab.plus.sum() / gross_product),
        ("нулевая цена", "строк", len(zero_p) / len(df)),
        ("qty<0 без C (складские)", "строк", len(neg_nonc) / len(df)),
        ("инвойс с >1 меткой времени", "инвойсов", multi_ts / len(inv)),
        ("клиент с >1 страной", "клиентов", (cc > 1).mean()),
        ("товар с >1 описанием", "кодов", (desc > 1).mean()),
    ],
    columns=["показатель", "база", "частота"],
)
rates["частота, %"] = (100 * rates["частота"]).round(2)
rates.drop(columns="частота")